# CodeBERT Devign training on Google Colab

Run all cells from top to bottom after selecting a GPU runtime. The first cell is intentionally self-contained because Colab sessions reset completely.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import os
import zipfile
import json
import sys

print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA GPU is required for practical CodeBERT fine-tuning. In Colab choose Runtime > Change runtime type > T4 GPU.')
    print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    torch = None
    print('Torch is not installed yet; the conditional install below will install it.')

required = {'transformers':'5.20.2','datasets':'5.1.0','scikit-learn':None,'accelerate':None,'torch':None}
def installed(name):
    try:
        import importlib.metadata as md
        return md.version(name)
    except md.PackageNotFoundError:
        return None
missing = [name for name, version in required.items() if installed(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.20.2', 'datasets==5.1.0', 'scikit-learn', 'accelerate', 'torch'])
print('Package check:', {name: installed(name) for name in required})

project_root = Path('/content/vuln-detect-codebert')
processed = project_root / 'data' / 'processed'
expected = [processed / name for name in ('train.jsonl','validation.jsonl','test.jsonl')]
missing_files = [str(path) for path in expected if not path.exists()]
print('Processed data present:', not missing_files, missing_files)


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise FileNotFoundError('Upload vuln-detect-codebert_export.zip or another project zip.')

extract_root = Path('/content/project_extract')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir()
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(extract_root)

matches = list(extract_root.rglob('src/train_codebert.py'))
if not matches:
    raise FileNotFoundError('Could not find src/train_codebert.py anywhere in the extracted zip; flat and wrapped layouts are both supported.')
project_root = matches[0].parent.parent
print('Project root:', project_root)
processed = project_root / 'data' / 'processed'
expected = [processed / name for name in ('train.jsonl','validation.jsonl','test.jsonl')]
if any(not path.exists() for path in expected):
    raise FileNotFoundError(f'Missing processed JSONL files: {[str(p) for p in expected if not p.exists()]}')
print('Verified:', [str(path) for path in expected])


In [ ]:
# Defensive re-check at the top of every fresh run.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Stop and select a GPU runtime.')
for path in (project_root / 'data' / 'processed' / name for name in ('train.jsonl','validation.jsonl','test.jsonl')):
    assert path.exists(), path
print('CUDA re-verified:', torch.cuda.get_device_name(0))
print('All three processed JSONL files re-verified.')
print(subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True).stdout)


In [ ]:
# Inspect the installed signatures before training.
nspace = {}
exec((project_root / 'src' / 'train_codebert.py').read_text(), nspace)
import inspect
from transformers import TrainingArguments, Trainer
print(inspect.signature(TrainingArguments))
print(inspect.signature(Trainer))


In [ ]:
# Run training from the project root.
os.chdir(project_root)
subprocess.check_call([sys.executable, 'src/train_codebert.py'])


In [ ]:
# Package the checkpoint and required metadata, then trigger the Colab download.
package_root = Path('/content/vuln-detect-codebert_package')
if package_root.exists(): shutil.rmtree(package_root)
package_root.mkdir()
for source, destination in [
    (project_root / 'models' / 'codebert', package_root / 'models' / 'codebert'),
    (project_root / 'data' / 'processed' / 'splits.json', package_root / 'data' / 'processed' / 'splits.json'),
    (project_root / 'reports' / 'baseline_results.json', package_root / 'reports' / 'baseline_results.json'),
]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir(): shutil.copytree(source, destination)
    else: shutil.copy2(source, destination)
archive = shutil.make_archive('/content/vuln-detect-codebert_colab_artifacts', 'gztar', package_root)
print('Created:', archive)
from google.colab import files
files.download(archive)
